# Hunyuan3D-2.1 — Shape-Only Headless API (Google Colab / T4 16GB)

Image → 3D mesh → `.glb`, exposed over a FastAPI server tunneled with ngrok.
No Gradio, no ComfyUI, no texture/PBR stage — shape generation only.

**Endpoints**
- `GET  /`
- `GET  /health`
- `POST /generate`   (multipart file upload, field name `file`)
- `GET  /status/{job_id}`
- `GET  /result/{job_id}`

**Before you run:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Put your ngrok authtoken in the config cell below (`NGROK_TOKEN`).
3. If you don't have a *reserved static domain* on your ngrok account, leave
   `NGROK_DOMAIN = None` and the notebook will use a random ephemeral URL instead
   (static domains require the domain to already be reserved in your ngrok dashboard;
   using an unreserved one will fail).

**What was fixed vs. the original single-file script**
- Removed a call to `pipeline.enable_model_cpu_offload(...)` — that's a *diffusers*
  method that does not exist on `Hunyuan3DDiTFlowMatchingPipeline`; it was silently
  failing into a fallback every single run.
- Removed the unsupported `dtype=` kwarg from `from_pretrained` (only `device`,
  `subfolder`, `use_safetensors` are supported by this custom pipeline).
- Fixed background-removal mode detection (previously missed palette/`L` images
  and mis-handled already-transparent images).
- Restored sane generation defaults (`num_inference_steps=20`, `guidance_scale=7.5`,
  `octree_resolution=256`, `num_chunks=20000`) — the original's `steps=5` was far
  below Tencent's own default of 30 and produces noticeably worse meshes.
- `PORT` config value is now actually threaded into the running server (was
  hardcoded to 8000 inside the generated server file).
- ngrok token is validated *before* the ~10 minute build/download, not after.
- Output filenames now include the job id, so two uploads with the same
  original filename can no longer overwrite each other.
- Added a fallback if the reserved ngrok domain isn't available on your account.


## 1. Config

In [ ]:
import os
import sys
import subprocess
import shutil
import time
import threading
import uuid
import traceback
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

REPO = Path("/content/Hunyuan3D-2.1")
VENV = Path("/content/hunyuan21_env")
SERVER_DIR = Path("/content/hunyuan21_server")

PYTHON = VENV / "bin" / "python"
PIP = VENV / "bin" / "pip"

PORT = 8000

# Set to None to just get a random ngrok URL each run (no reserved domain needed).
NGROK_DOMAIN = "jaybird-heroic-garfish.ngrok-free.app"

# ============================================================
# ใส่ NGROK TOKEN ตรงนี้
# ============================================================

NGROK_TOKEN = "ใส่_TOKEN_ของคุณ"

# Fail fast, before doing 10+ minutes of setup work.
if not NGROK_TOKEN or NGROK_TOKEN == "ใส่_TOKEN_ของคุณ":
    raise RuntimeError("ใส่ NGROK_TOKEN ที่ CONFIG ก่อนรัน (set NGROK_TOKEN before running)")

print("Config OK. Python:", sys.version)


## 2. Helpers

In [ ]:
def run(args, cwd=None, env=None):
    args = [str(x) for x in args]
    print("\n>>>", " ".join(args))
    result = subprocess.run(args, cwd=str(cwd) if cwd else None, env=env)
    if result.returncode != 0:
        raise RuntimeError(
            "\nCOMMAND FAILED:\n" + " ".join(args) + "\nRETURN CODE: " + str(result.returncode)
        )
    return result


def run_bash(script, env=None):
    print("\n>>> bash script")
    result = subprocess.run(["bash", "-lc", script], env=env)
    if result.returncode != 0:
        raise RuntimeError(f"\nBASH COMMAND FAILED\nRETURN CODE: {result.returncode}")
    return result


## 3. GPU check

In [ ]:
run(["nvidia-smi"])


## 4. System packages

In [ ]:
run_bash("""
apt-get update -qq

apt-get install -y -qq \
    git \
    wget \
    curl \
    ffmpeg \
    build-essential \
    cmake \
    ninja-build \
    libgl1 \
    libglib2.0-0 \
    libjpeg-dev \
    pkg-config \
    python3.10 \
    python3.10-dev \
    python3.10-venv
""")


## 5. Python 3.10 virtual environment

In [ ]:
if VENV.exists():
    print("\nRemoving old Hunyuan environment...")
    shutil.rmtree(VENV, ignore_errors=True)

print("\nCreating Python 3.10 venv...")
run(["python3.10", "-m", "venv", str(VENV)])
run([PYTHON, "--version"])

run([PYTHON, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])


## 6. Clone the official Tencent-Hunyuan repo

In [ ]:
if REPO.exists():
    print("\nOfficial repository already exists.")
    run(["git", "pull"], cwd=REPO)
else:
    print("\nCloning official Tencent repository...")
    run([
        "git", "clone", "--recurse-submodules",
        "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git",
        str(REPO)
    ])

run(["git", "submodule", "update", "--init", "--recursive"], cwd=REPO)


## 7. PyTorch 2.5.1 + CUDA 12.4

This is the combination Hunyuan3D-2.1 was tested against.

In [ ]:
run([
    PYTHON, "-m", "pip", "install",
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu124"
])

torch_test = r"""
import torch
print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print("GPU", i, ":", p.name)
        print("VRAM:", round(p.total_memory / 1024**3, 2), "GB")
"""
run([PYTHON, "-c", torch_test])


## 8. Shape-only dependencies

Only `hy3dshape/requirements.txt` is installed. Paint/texture (`hy3dpaint`) deps,
including the `custom_rasterizer` CUDA extension, are intentionally skipped — they
are only required for the PBR texture stage, not for shape generation.

In [ ]:
REQ_FILE = REPO / "hy3dshape" / "requirements.txt"

requirements = []
for line in REQ_FILE.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    name = line.lower()
    if name.startswith("torch") or name.startswith("torchvision") or name.startswith("torchaudio"):
        continue
    requirements.append(line)

run([PYTHON, "-m", "pip", "install", *requirements])

run([
    PYTHON, "-m", "pip", "install", "-U",
    "accelerate",
    "huggingface_hub",
    "fastapi",
    "uvicorn",
    "python-multipart",
    "pyngrok",
    "requests",
    "rembg",
    "onnxruntime"
])

runtime_env = os.environ.copy()
runtime_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
runtime_env["HF_HOME"] = "/content/huggingface"
runtime_env["TRANSFORMERS_CACHE"] = "/content/huggingface"
runtime_env["HUNYUAN_PORT"] = str(PORT)


## 9. Import sanity check

In [ ]:
import_test = r"""
import sys
sys.path.insert(0, "/content/Hunyuan3D-2.1/hy3dshape")

import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

import rembg
print("rembg: OK")

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
print("Hunyuan3DDiTFlowMatchingPipeline: OK")

from hy3dshape.rembg import BackgroundRemover
print("BackgroundRemover: OK")
"""
run([PYTHON, "-c", import_test], cwd=REPO, env=runtime_env)


## 10. Write the FastAPI server

In [ ]:
SERVER_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR = SERVER_DIR / "uploads"
OUTPUT_DIR = SERVER_DIR / "outputs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SERVER_FILE = SERVER_DIR / "server.py"

SERVER_CODE = r'''
import os
import sys
import uuid
import threading
import traceback
from pathlib import Path

sys.path.insert(0, "/content/Hunyuan3D-2.1/hy3dshape")

import torch
from PIL import Image

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
from hy3dshape.rembg import BackgroundRemover

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

# ============================================================
# PATHS
# ============================================================

BASE = Path("/content/hunyuan21_server")
UPLOADS = BASE / "uploads"
OUTPUTS = BASE / "outputs"
UPLOADS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

PORT = int(os.environ.get("HUNYUAN_PORT", "8000"))

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ============================================================
# JOB STATE
# ============================================================

JOBS = {}
JOBS_LOCK = threading.Lock()
GPU_LOCK = threading.Lock()


def set_job(job_id, **data):
    with JOBS_LOCK:
        JOBS.setdefault(job_id, {})
        JOBS[job_id].update(data)


def get_job(job_id):
    with JOBS_LOCK:
        return dict(JOBS.get(job_id, {}))


# ============================================================
# MODEL
# ============================================================

print()
print("=" * 70)
print("LOADING HUNYUAN3D-SHAPE-V2-1")
print("=" * 70)

MODEL_NAME = "tencent/Hunyuan3D-2.1"

# NOTE: `dtype=` is not a supported kwarg on this custom pipeline (only
# `device`, `subfolder`, `use_safetensors` are). Weights load fp16 by default.
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    MODEL_NAME,
    device="cuda",
)

# NOTE: there is no `enable_model_cpu_offload` on this pipeline (that is a
# diffusers-only API). The shape stage alone fits comfortably in a T4's 16GB
# (~10GB per Tencent's own docs), so no offload is needed here.

try:
    bg_remover = BackgroundRemover()
    print("Background remover: READY")
except Exception as error:
    bg_remover = None
    print("Background remover unavailable:", error)

print("=" * 70)
print("HUNYUAN3D MODEL READY")
print("=" * 70)

# ============================================================
# API
# ============================================================

app = FastAPI(title="Hunyuan3D-2.1 Headless API", version="1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
def root():
    return {
        "status": "online",
        "model": MODEL_NAME,
        "type": "image-to-3d-shape",
        "vram": "10GB recommended",
        "endpoint": "/generate",
    }


@app.get("/health")
def health():
    return {
        "status": "ok",
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }


@app.post("/generate")
async def generate(file: UploadFile = File(...)):
    if not file.filename:
        raise HTTPException(status_code=400, detail="Filename missing")

    filename = Path(file.filename).name
    job_id = uuid.uuid4().hex[:12]

    input_path = UPLOADS / f"{job_id}_{filename}"

    output_stem = Path(filename).stem.strip()
    if not output_stem:
        output_stem = "model_" + job_id

    # job_id embedded so two uploads with the same original filename
    # cannot overwrite each other.
    output_filename = f"{output_stem}_{job_id}.glb"
    output_path = OUTPUTS / output_filename

    with open(input_path, "wb") as f:
        while True:
            chunk = await file.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)

    set_job(
        job_id,
        status="queued",
        filename=output_filename,
        input_filename=filename,
    )

    threading.Thread(
        target=process_job,
        args=(job_id, input_path, output_path),
        daemon=True,
    ).start()

    return {"job_id": job_id, "status": "queued", "filename": output_filename}


# Loads the image and runs background removal only when the image does
# not already carry usable transparency.
def prepare_image(input_path):
    image = Image.open(input_path)
    image.load()

    has_alpha = image.mode in ("RGBA", "LA") or (
        image.mode == "P" and "transparency" in image.info
    )

    if has_alpha:
        return image.convert("RGBA")

    image = image.convert("RGB")

    if bg_remover is not None:
        try:
            image = bg_remover(image)
        except Exception as error:
            print("Background removal failed:", error)

    return image


def process_job(job_id, input_path, output_path):
    try:
        with GPU_LOCK:
            set_job(job_id, status="loading")

            image = prepare_image(input_path)

            set_job(job_id, status="generating")

            # Tencent's documented defaults: num_inference_steps=30,
            # guidance_scale=7.5, octree_resolution=256. Steps trimmed to 20
            # for a good quality/speed balance on a T4; the rest kept at the
            # documented defaults rather than the far more aggressive (and
            # much lower quality) values used previously.
            with torch.inference_mode():
                mesh = pipeline(
                    image=image,
                    num_inference_steps=20,
                    guidance_scale=7.5,
                    octree_resolution=256,
                    num_chunks=20000,
                    output_type="trimesh",
                    enable_pbar=True,
                )[0]

            set_job(job_id, status="exporting")

            mesh.export(str(output_path))

            del mesh
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            set_job(
                job_id,
                status="completed",
                filename=output_path.name,
                output=str(output_path),
            )

            print("\nJOB COMPLETE:", job_id)
            print("OUTPUT:", output_path)

    except Exception as error:
        traceback.print_exc()
        set_job(
            job_id,
            status="error",
            error=str(error),
            traceback=traceback.format_exc(),
        )
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

    finally:
        try:
            input_path.unlink()
        except Exception:
            pass


@app.get("/status/{job_id}")
def status(job_id: str):
    data = get_job(job_id)
    if not data:
        raise HTTPException(status_code=404, detail="Job not found")
    return data


@app.get("/result/{job_id}")
def result(job_id: str):
    data = get_job(job_id)

    if not data:
        raise HTTPException(status_code=404, detail="Job not found")

    if data.get("status") != "completed":
        raise HTTPException(
            status_code=409,
            detail={"status": data.get("status"), "error": data.get("error")},
        )

    output = data.get("output")
    if not output or not Path(output).exists():
        raise HTTPException(status_code=404, detail="Output file missing")

    return FileResponse(
        output,
        media_type="model/gltf-binary",
        filename=data.get("filename", "model.glb"),
    )


import uvicorn
uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")
'''

SERVER_FILE.write_text(SERVER_CODE)
print("Server written to:", SERVER_FILE)


## 11. Start the API server

In [ ]:
import requests

server = subprocess.Popen(
    [str(PYTHON), str(SERVER_FILE)],
    cwd=str(REPO),
    env=runtime_env,
)

server_ready = False
for _ in range(300):
    try:
        response = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2)
        if response.status_code == 200:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not server_ready:
    if server.poll() is not None:
        raise RuntimeError("Hunyuan3D server exited during startup. Check the cell output above for the traceback.")
    raise RuntimeError("Hunyuan3D server did not become ready within 300s.")

print("Hunyuan3D API: READY")


## 12. Expose it with ngrok

In [ ]:
from pyngrok import ngrok, conf

try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(NGROK_TOKEN)

print("\nStarting ngrok...")

tunnel = None
if NGROK_DOMAIN:
    try:
        tunnel = ngrok.connect(addr=PORT, proto="http", domain=NGROK_DOMAIN)
    except Exception as error:
        print(f"Could not bind reserved domain '{NGROK_DOMAIN}': {error}")
        print("Falling back to a random ephemeral ngrok URL.")

if tunnel is None:
    tunnel = ngrok.connect(addr=PORT, proto="http")

PUBLIC_URL = str(tunnel.public_url)
if PUBLIC_URL.startswith("http://"):
    PUBLIC_URL = "https://" + PUBLIC_URL[7:]

print()
print("=" * 70)
print("HUNYUAN3D-2.1 HEADLESS ONLINE")
print("=" * 70)
print()
print("PUBLIC URL:", PUBLIC_URL)
print("HEALTH:    ", PUBLIC_URL + "/health")
print("GENERATE:  ", PUBLIC_URL + "/generate")
print("STATUS:    ", PUBLIC_URL + "/status/{job_id}")
print("RESULT:    ", PUBLIC_URL + "/result/{job_id}")
print()
print("MODEL: tencent/Hunyuan3D-2.1")
print("MODE:  Single image -> 3D shape -> GLB")
print("=" * 70)


## 13. Keep the Colab runtime alive

Run this last. It blocks and prints a heartbeat every 30s until you interrupt
the cell (Runtime → Interrupt execution) or the server process dies.

In [ ]:
try:
    while True:
        if server.poll() is not None:
            print("Hunyuan3D server stopped.")
            break
        time.sleep(30)
        print("[HUNYUAN3D] ONLINE |", PUBLIC_URL)
except KeyboardInterrupt:
    print("Stopping...")
    server.terminate()


## Usage example (run from your own machine, not in this notebook)

```bash
# submit a job
curl -X POST "$PUBLIC_URL/generate" -F "file=@car.png"
# -> {"job_id": "...", "status": "queued", "filename": "car_XXXXXXXXXXXX.glb"}

# poll status
curl "$PUBLIC_URL/status/<job_id>"

# download once status == "completed"
curl -OJ "$PUBLIC_URL/result/<job_id>"
```
